# Lesson 03 - Agentic Design Patterns

Σε αυτό το μάθημα, εξερευνούμε τρία βασικά πρότυπα σχεδίασης για την κατασκευή αποτελεσματικών πρακτόρων AI:

1. **Σαφείς Οδηγίες για τον Πράκτορα** — Δημιουργία ακριβών, οριστικών ρόλου prompts που καθοδηγούν τη συμπεριφορά του πράκτορα  
2. **Δομημένη Έξοδος με Μοντέλα Pydantic** — Εξασφάλιση ότι οι πράκτορες επιστρέφουν προβλέψιμα, επικυρωμένα δεδομένα  
3. **Πράκτορες Μοναδικής Ευθύνης** — Σχεδιασμός εστιασμένων πρακτόρων που κάνουν καλά το κάθε τι από ένα πράγμα  

Θα εφαρμόσουμε κάθε πρότυπο σε ένα σενάριο **συστήματος πρότασης προορισμών ταξιδιών**, χτίζοντας σταδιακά ένα σύστημα που μπορεί να προτείνει προορισμούς, να ελέγχει τη διαθεσιμότητα και να διαχειρίζεται τη λογιστική.


## Ρύθμιση


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Πρότυπο 1: Σαφείς Οδηγίες προς τον Πράκτορα

Το πιο αποτελεσματικό πρότυπο είναι επίσης το απλούστερο: να γράφετε σαφείς, λεπτομερείς οδηγίες για τον πράκτορά σας.

Οι καλές οδηγίες ορίζουν:
- **Ποιος** είναι ο πράκτορας (προσωπικότητα και τόνος)
- **Τι** πρέπει να κάνει (ευθύνες βήμα προς βήμα)
- **Πώς** πρέπει να συμπεριφέρεται (περιορισμοί και στυλ)

Παρακάτω, δημιουργούμε έναν ταξιδιωτικό concierge πράκτορα με ρητές οδηγίες που διαμορφώνουν κάθε απάντηση που παράγει.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Πρότυπο 2: Δομημένη Έξοδος με Μοντέλα Pydantic

Το ελεύθερο κείμενο είναι χρήσιμο για συζήτηση, αλλά τα κατώτερα συστήματα χρειάζονται δομημένα δεδομένα.  
Συνδυάζοντας **μοντέλα Pydantic** με μια **λειτουργία εργαλείου**, μπορούμε να:

- Ορίσουμε ένα ακριβές σχήμα για την έξοδο του πρακτορείου  
- Επαληθεύσουμε τις απαντήσεις αυτόματα  
- Ενσωματώσουμε με αξιοπιστία τα αποτελέσματα του πρακτορείου στην λογική της εφαρμογής  

Εισάγουμε επίσης ένα εργαλείο που επιστρέφει λεπτομέρειες προορισμού ώστε ο πράκτορας να βασίζει τις συστάσεις του σε πραγματικά δεδομένα.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Πρότυπο 3: Πράκτορες Μοναδικής Ευθύνης

Οι πολύπλοκες εργασίες ωφελούνται από το διαμερισμό της εργασίας σε πολλούς εστιασμένους πράκτορες, ο καθένας με μοναδική ευθύνη:

- Ένας **Ειδικός Προορισμού** που γνωρίζει για μέρη και διαθεσιμότητα
- Ένας **Σχεδιαστής logistics** που χειρίζεται πτήσεις, ξενοδοχεία και δρομολόγια

Αυτό αντικατοπτρίζει την αρχή της μηχανικής λογισμικού της *διαχωρισμού των ανησυχιών* — κάθε πράκτορας είναι πιο εύκολος στο να δοκιμαστεί, να συντηρηθεί και να βελτιωθεί ανεξάρτητα.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Περίληψη

Σε αυτό το μάθημα εφαρμόσαμε τρία πρότυπα σχεδίασης πράκτορα σε ένα σενάριο συστάσεων ταξιδιών:

| Πρότυπο | Κύρια Ιδέα | Οφέλη |
|---|---|---|
| **Σαφείς Οδηγίες** | Ορίστε την περσόνα, τις ευθύνες και τους περιορισμούς εκ των προτέρων | Συνεπής, σύμφωνα με το brand συμπεριφορά πράκτορα |
| **Δομημένη Έξοδος** | Χρησιμοποιήστε τα μοντέλα Pydantic ως μορφή απάντησης | Επικυρωμένα, αναγνώσιμα από μηχανή αποτελέσματα |
| **Μοναδική Ευθύνη** | Αναθέστε σε κάθε πράκτορα μία εστιασμένη εργασία | Ευκολότερος έλεγχος, συντήρηση και σύνθεση |

Αυτά τα πρότυπα συνδυάζονται φυσικά — μπορείτε να συνδυάσετε σαφείς οδηγίες με δομημένη έξοδο μέσα σε έναν πράκτορα με μοναδική ευθύνη για να δημιουργήσετε ανθεκτικά, έτοιμα για παραγωγή συστήματα.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:  
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία αυτόματης μετάφρασης AI [Co-op Translator](https://github.com/Azure/co-op-translator). Παρόλο που επιδιώκουμε την ακρίβεια, παρακαλούμε να λάβετε υπόψη ότι οι αυτόματες μεταφράσεις ενδέχεται να περιέχουν σφάλματα ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η επίσημη πηγή. Για κρίσιμες πληροφορίες συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για οποιεσδήποτε παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
